In [73]:
# Setup and helper functions
from SPARQLWrapper import SPARQLWrapper, JSON
import re
import time
from urllib.parse import unquote
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL
from tqdm import tqdm

DBPEDIA_SPARQL = "https://dbpedia.org/sparql"
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

dbp_sparql = SPARQLWrapper(DBPEDIA_SPARQL)
dbp_sparql.setReturnFormat(JSON)

wd_sparql = SPARQLWrapper(WIKIDATA_SPARQL)
wd_sparql.setReturnFormat(JSON)
wd_sparql.addParameter("timeout", "300000")

EX = Namespace("http://example.org/travelkg/")

CATEGORY_REGEX = re.compile(r"^http://dbpedia\.org/resource/Category:(.+?)_(in|of)_(.+)")

def get_visitor_attraction_categories(limit=100):
    query = f"""
    SELECT DISTINCT ?category WHERE {{
      ?category a skos:Concept .
      FILTER regex(str(?category), "^http://dbpedia.org/resource/Category:Tourist_attractions_in_")
    }} LIMIT {limit}
    """
    dbp_sparql.setQuery(query)
    res = dbp_sparql.query().convert()
    return [r["category"]["value"] for r in res["results"]["bindings"]]

def get_pois_for_category(category_uri):
    query = f"""
    SELECT DISTINCT ?POI ?category WHERE {{
      ?POI <http://purl.org/dc/terms/subject> ?category .
      ?category <http://www.w3.org/2004/02/skos/core#broader> <{category_uri}> .
    }}
    """
    dbp_sparql.setQuery(query)
    res = dbp_sparql.query().convert()
    pois = []
    for r in res["results"]["bindings"]:
        pois.append((r["POI"]["value"], r["category"]["value"]))
    return pois

# def get_pois_for_category(category_uri):
#     """Return POIs whose subject category is a child of category_uri, excluding subject categories that are themselves Tourist_attractions_*"""
#     query = f"""
#     SELECT DISTINCT ?POI ?category WHERE {{
#       ?POI <http://purl.org/dc/terms/subject> ?category .
#       ?category <http://www.w3.org/2004/02/skos/core#broader> <{category_uri}> .
#       FILTER(!regex(str(?category), "Category:Tourist_attractions", "i"))
#     }}
#     """
#     dbp_sparql.setQuery(query)
#     res = dbp_sparql.query().convert()
#     return [(r["POI"]["value"], r["category"]["value"]) for r in res.get("results", {}).get("bindings", [])]

def parse_category_uri(category_uri):
    m = CATEGORY_REGEX.match(category_uri)
    if not m:
        return None, None
    type_str = m.group(1).replace("_", " ")
    location_str = m.group(3).replace("_", " ")
    return type_str, location_str

def canonical_dbpedia_resource(uri_or_page):
    if uri_or_page is None:
        return None
    s = unquote(uri_or_page.strip())
    if re.match(r"^https?://dbpedia\.org/resource/", s):
        return s
    m = re.match(r"^https?://dbpedia\.org/page/(.+)$", s)
    if m:
        return f"http://dbpedia.org/resource/{m.group(1)}"
    if not s.startswith("http://") and not s.startswith("https://"):
        return f"http://dbpedia.org/resource/{s}"
    return s

def get_wikidata_mapping(dbpedia_poi_uri):
    resource = canonical_dbpedia_resource(dbpedia_poi_uri)
    if resource is None:
        return []
    query = f"""
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    SELECT DISTINCT ?wikidata WHERE {{
      <{resource}> owl:sameAs ?wikidata .
      FILTER(STRSTARTS(STR(?wikidata), "http://www.wikidata.org/entity/") || STRSTARTS(STR(?wikidata), "https://www.wikidata.org/entity/"))
    }}
    """
    dbp_sparql.setQuery(query)
    try:
        res = dbp_sparql.query().convert()
    except Exception as e:
        print("DBpedia mapping error", e)
        return []
    return [r["wikidata"]["value"] for r in res.get("results", {}).get("bindings", []) if r.get("wikidata")]

def get_location_data_wikidata(wd_uri):
    entity = wd_uri.replace("http://www.wikidata.org/entity/", "wd:")
    query = f"""
    SELECT ?countryLabel ?admin ?adminLabel WHERE {{
      OPTIONAL {{ {entity} wdt:P17 ?country . }}
      OPTIONAL {{ {entity} wdt:P131 ?admin . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    wd_sparql.setQuery(query)
    try:
        res = wd_sparql.query().convert()
    except Exception as e:
        print("Wikidata location error", e)
        return {"country": None, "admin": None}
    country = None
    admin = None
    for b in res.get("results", {}).get("bindings", []):
        if not country and "countryLabel" in b:
            country = b["countryLabel"]["value"]
        if "adminLabel" in b:
            admin = b["adminLabel"]["value"]
    return {"country": country, "admin": admin}

def get_image_wikidata(wd_uri):
    query = f"""
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    SELECT ?img WHERE {{
      <{wd_uri}> wdt:P18 ?img .
    }} LIMIT 1
    """
    wd_sparql.setQuery(query)
    try:
        res = wd_sparql.query().convert()
    except Exception as e:
        print("Wikidata image error", e)
        return None
    bindings = res.get("results", {}).get("bindings", [])
    if not bindings:
        return None
    return bindings[0].get("img", {}).get("value")



In [77]:
# Step 1: fetch categories, POIs, enrich with wikidata mapping, location, image

def build_base_df(category_limit=100, sleep_between_wd=0.05):
    categories = get_visitor_attraction_categories(limit=category_limit)
    print(f"Found {len(categories)} categories")
    records = []
    for cat_uri in tqdm(categories, desc="Categories"):
        for poi_uri, category_uri in get_pois_for_category(cat_uri):
            type_str, location_str = parse_category_uri(category_uri)
            if not type_str or not location_str:
                continue
            records.append({
                "POI": poi_uri,
                "Category": category_uri,
                "Type": type_str,
                "Location": location_str,
            })

    df = pd.DataFrame(records).drop_duplicates(subset=["POI", "Category"])
    df["Wikidata_entity"] = None
    df["Country"] = None
    df["AdminTerritory"] = None
    df["Image"] = None

    wd_cache = {}
    loc_cache = {}
    img_cache = {}

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Enrich POIs"):
        poi = row["POI"]
        if poi in wd_cache:
            wd_entities = wd_cache[poi]
        else:
            wd_entities = get_wikidata_mapping(poi)
            wd_cache[poi] = wd_entities
            time.sleep(sleep_between_wd)

        wd_entity = wd_entities[0] if wd_entities else None
        df.at[idx, "Wikidata_entity"] = wd_entity

        if wd_entity:
            if wd_entity in loc_cache:
                loc = loc_cache[wd_entity]
            else:
                loc = get_location_data_wikidata(wd_entity)
                loc_cache[wd_entity] = loc
                time.sleep(sleep_between_wd)
            df.at[idx, "Country"] = loc.get("country")
            df.at[idx, "AdminTerritory"] = loc.get("admin")

            if wd_entity in img_cache:
                img = img_cache[wd_entity]
            else:
                img = get_image_wikidata(wd_entity)
                img_cache[wd_entity] = img
                time.sleep(sleep_between_wd)
            df.at[idx, "Image"] = img

        time.sleep(0.01)

    return df

# build global df
# try:
#     df  # reuse if already exists
# except NameError:
df = build_base_df(category_limit=20)

df.head()



Found 20 categories


Enrich POIs:  24%|██▎       | 227/964 [04:06<1:09:32,  5.66s/it]

DBpedia mapping error HTTP Error 502: Bad Gateway
DBpedia mapping error HTTP Error 502: Bad Gateway


Enrich POIs:  24%|██▎       | 228/964 [04:06<49:04,  4.00s/it]  

DBpedia mapping error HTTP Error 502: Bad Gateway
DBpedia mapping error HTTP Error 502: Bad Gateway


...

DBpedia mapping error HTTP Error 502: Bad Gateway
DBpedia mapping error HTTP Error 502: Bad Gateway


Enrich POIs:  95%|█████████▌| 916/964 [10:48<00:06,  7.92it/s]

DBpedia mapping error HTTP Error 502: Bad Gateway


Enrich POIs: 100%|██████████| 964/964 [11:52<00:00,  1.35it/s]


,POI,Category,Type,Location,Wikidata_entity,Country,AdminTerritory,Image
0,http://dbpedia.org/resource/Sam_Houston_Jones_...,http://dbpedia.org/resource/Category:Protected...,Protected areas,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q7407643,United States,Louisiana,http://commons.wikimedia.org/wiki/Special:File...
1,http://dbpedia.org/resource/Cathedral_of_the_I...,http://dbpedia.org/resource/Category:Churches_...,Churches,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q5052447,United States,Louisiana,http://commons.wikimedia.org/wiki/Special:File...
2,http://dbpedia.org/resource/All_Saints_Episcop...,http://dbpedia.org/resource/Category:Churches_...,Churches,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q4729565,United States,DeQuincy,http://commons.wikimedia.org/wiki/Special:File...
3,http://dbpedia.org/resource/Church_of_the_Good...,http://dbpedia.org/resource/Category:Churches_...,Churches,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q5117862,United States,Louisiana,None
4,http://dbpedia.org/resource/USS_Orleck,http://dbpedia.org/resource/Category:Museums_i...,Museums,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q7872490,None,None,http://commons.wikimedia.org/wiki/Special:File...


In [ ]:
# Step 2: interest mapping (keyword-only)

INTEREST_KEYWORDS = {
    "Culture": [
        "museum", "heritage", "cultural", "monument", "memorial",
        "libraries", "universities and colleges",
        "national trust properties", "squares", "houses"
    ],
    "Art": ["art", "gallery", "exhibit"],
    "History": [
        "history", "historic", "castle", "palace", "ruin",
        "roman sites", "archaeological sites", "cemeteries",
        "national trust properties", "windmills", "towers", "lighthouses",
        "canals"
    ],
    "Nature": [
        "park", "beach", "lake", "garden", "forest", "mountain",
        "protected areas", "nature reserves", "footpaths", "piers",
        "lighthouses"
    ],
    "Sports": ["stadium", "arena", "sports"],
    "Religion": ["church", "cathedral", "mosque", "temple"],
    "Entertainment": [
        "theatre", "concert", "music", "amusement", "casino",
        "cinemas", "festivals", "entertainment venues"
    ],
}

def apply_interest_mapping(df_in, keyword_map=None):
    """Assign interests purely from the keyword map; no extra overrides."""
    kw_map = keyword_map or INTEREST_KEYWORDS
    df_out = df_in.copy()
    labels = (df_out["Type"].fillna("") + " " + df_out["Category"].fillna("")).str.lower()
    interests_col = []
    for text in labels:
        interests = []
        for interest, kws in kw_map.items():
            if any(kw in text for kw in kws):
                interests.append(interest)
        interests_col.append(sorted(set(interests)))
    df_out["Interests"] = interests_col
    return df_out

# build interest-augmented df (optional; rerun when keyword map changes)
df_with_interests = apply_interest_mapping(df)
df_with_interests.head()



,POI,Category,Type,Location,Wikidata_entity,Country,AdminTerritory,Image,Interests
0,http://dbpedia.org/resource/Sam_Houston_Jones_...,http://dbpedia.org/resource/Category:Protected...,Protected areas,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q7407643,United States,Louisiana,http://commons.wikimedia.org/wiki/Special:File...,[Nature]
1,http://dbpedia.org/resource/Cathedral_of_the_I...,http://dbpedia.org/resource/Category:Churches_...,Churches,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q5052447,United States,Louisiana,http://commons.wikimedia.org/wiki/Special:File...,[Religion]
2,http://dbpedia.org/resource/All_Saints_Episcop...,http://dbpedia.org/resource/Category:Churches_...,Churches,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q4729565,United States,DeQuincy,http://commons.wikimedia.org/wiki/Special:File...,[Religion]
3,http://dbpedia.org/resource/Church_of_the_Good...,http://dbpedia.org/resource/Category:Churches_...,Churches,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q5117862,United States,Louisiana,None,[Religion]
4,http://dbpedia.org/resource/USS_Orleck,http://dbpedia.org/resource/Category:Museums_i...,Museums,"Calcasieu Parish, Louisiana",http://www.wikidata.org/entity/Q7872490,None,None,http://commons.wikimedia.org/wiki/Special:File...,[Culture]


In [ ]:
# Rows with exactly one interest
df_one_interest = df_with_interests[df_with_interests['Interest'].str.len() == 0]
df_one_interest

,POI,Category,Type,Location,Wikidata_entity,Country,AdminTerritory,Image,Interests
61,http://dbpedia.org/resource/Portland_Observatory,http://dbpedia.org/resource/Category:Tourist_a...,Tourist attractions,"Portland, Maine",http://www.wikidata.org/entity/Q2105180,United States,Portland,http://commons.wikimedia.org/wiki/Special:File...,[]


In [ ]:
# Step 3: build knowledge graph (uses interests already attached to df)

def build_graph(df_in, out_ttl="visitor_attractions.ttl"):
    g = Graph()
    g.bind("ex", EX)
    g.bind("rdfs", RDFS)
    g.bind("owl", OWL)

    interest_nodes = {
    "museum": {"Culture", "Art", "History"},
    "art museum": {"Art", "Culture"},
    "gallery": {"Art", "Culture"},
    "heritage site": {"Culture", "History"},
    "park": {"Nature"},
    "national park": {"Nature"},
    "beach": {"Nature"},
    "garden": {"Nature"},
    "zoo": {"Nature"},
    "church": {"Religion", "Culture", "History"},
    "cathedral": {"Religion", "Culture", "History"},
    "mosque": {"Religion", "Culture", "History"},
    "temple": {"Religion", "Culture", "History"},
    "palace": {"History", "Architecture", "Culture"},
    "castle": {"History", "Culture", "Architecture"},
    "memorial": {"History", "Culture"},
    "theatre": {"Entertainment", "Culture"},
    "stadium": {"Sports"},
    }
    
    for interest in set(sum(df_in["Interests"].apply(lambda x: x or []).tolist(), [])):
        node = EX[interest.replace(" ", "_")]
        interest_nodes[interest] = node
        g.add((node, RDFS.label, Literal(interest)))

    for _, row in df_in.iterrows():
        poi_uri = URIRef(canonical_dbpedia_resource(row["POI"]))
        type_label = row.get("Type") or "Unknown_Type"
        type_node = EX[type_label.replace(" ", "_")]
        g.add((poi_uri, RDF.type, type_node))
        g.add((type_node, RDFS.label, Literal(type_label)))

        loc_label = row.get("Location")
        if loc_label:
            loc_node = EX[loc_label.replace(" ", "_")]
            g.add((poi_uri, EX.hasLocation, loc_node))
            g.add((loc_node, RDFS.label, Literal(loc_label)))

        if row.get("Wikidata_entity"):
            g.add((poi_uri, OWL.sameAs, URIRef(row["Wikidata_entity"])))

        if row.get("Image"):
            g.add((poi_uri, EX.image, Literal(row["Image"])))

        if row.get("AdminTerritory"):
            admin_node = EX[row["AdminTerritory"].replace(" ", "_")]
            g.add((poi_uri, EX.adminTerritory, admin_node))
            g.add((admin_node, RDFS.label, Literal(row["AdminTerritory"])))

        if row.get("Country"):
            country_node = EX[row["Country"].replace(" ", "_")]
            g.add((poi_uri, EX.country, country_node))
            g.add((country_node, RDFS.label, Literal(row["Country"])))

        for interest in row.get("Interests") or []:
            interest_node = interest_nodes.get(interest)
            if interest_node:
                g.add((type_node, RDFS.subClassOf, interest_node))

    g.serialize(destination=out_ttl, format="turtle")
    print(f"Saved KG to {out_ttl} with {len(g)} triples")
    return g

# Uncomment to write the graph
# graph = build_graph(df_with_interests)



In [ ]:
graph = build_graph(df_with_interests)

Saved KG to visitor_attractions.ttl with 3608 triples
